**TSNE mathematical visualization: Generated By Gemini, Edited and Prompted By Manim Community Nepal**

# Mathematical Notes: t-Distributed Stochastic Neighbor Embedding (t-SNE)

## 1. Prerequisites
To understand t-SNE, you need familiarity with the following concepts from probability and information theory.

### A. Conditional & Joint Probability
* **Conditional Probability $P(A|B)$:** The probability of event A occurring given that B is true.
* **Joint Probability $P(A, B)$:** The probability of both events occurring.

### B. The Normal (Gaussian) Distribution
Used to model similarity in the **High-Dimensional** space.
The probability density function (PDF) for a variable $x$ given mean $\mu$ and variance $\sigma^2$ is:

$$
f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}
$$

In t-SNE, we care about the decay profile: $e^{-\|x_i - x_j\|^2 / 2\sigma^2}$.

### C. The Student's t-Distribution (Cauchy Distribution)
Used to model similarity in the **Low-Dimensional** space.
Specifically, a t-distribution with **1 degree of freedom** (also known as the Cauchy distribution). Its PDF is:

$$
f(y) \propto \frac{1}{1 + y^2}
$$

**Key property:** It has **heavier tails** than a Gaussian. This means distant points have a higher probability than they would in a Gaussian distribution, which helps solve the "Crowding Problem."

### D. Kullback-Leibler (KL) Divergence
A measure of how one probability distribution $Q$ differs from a reference distribution $P$.

$$
KL(P || Q) = \sum_{i} p_i \log \frac{p_i}{q_i} = \sum_{i} p_i (\log p_i - \log q_i)
$$

* **Minimizing KL Divergence:** We want $P$ (high-D structure) and $Q$ (low-D structure) to be as similar as possible.
* Note: It is asymmetric. $KL(P||Q) \neq KL(Q||P)$.

---

## 2. Mathematical Formulation

### Step 1: High-Dimensional Similarities ($P$)
Given a dataset $X = \{x_1, x_2, \dots, x_N\}$ where $x_i \in \mathbb{R}^D$.

We first compute the conditional probability $p_{j|i}$ that $x_i$ would pick $x_j$ as its neighbor if neighbors were picked in proportion to their probability density under a Gaussian centered at $x_i$.

$$
p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|x_i - x_k\|^2 / 2\sigma_i^2)}
$$

**The Sigma ($\sigma_i$):**
The bandwidth $\sigma_i$ is determined for each point individually to maintain a fixed **Perplexity**.

$$
\text{Perplexity}(P_i) = 2^{H(P_i)}
$$

Where $H(P_i)$ is the Shannon entropy. This adapts to density: dense regions get smaller $\sigma_i$, sparse regions get larger $\sigma_i$.

**Symmetrization:**
We define the joint probabilities $p_{ij}$ (symmetric similarity):

$$
p_{ij} = \frac{p_{j|i} + p_{i|j}}{2N}
$$

This ensures $\sum_{i,j} p_{ij} = 1$ and makes $p_{ij} = p_{ji}$.

### Step 2: Low-Dimensional Similarities ($Q$)
We map points to $Y = \{y_1, y_2, \dots, y_N\}$ where $y_i \in \mathbb{R}^d$ (usually $d=2$).

We use the Student's t-distribution (without $\sigma$) for similarities:

$$
q_{ij} = \frac{(1 + \|y_i - y_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|y_k - y_l\|^2)^{-1}}
$$

Note: The numerator $(1 + \|y_i - y_j\|^2)^{-1}$ is the unnormalized kernel. Let's call the denominator $Z$ (partition function).

### Step 3: Cost Function
We optimize the positions $y_i$ by minimizing the KL divergence between $P$ and $Q$:

$$
C = KL(P||Q) = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}
$$

---

## 3. Derivation of the Gradient
To minimize $C$ using Gradient Descent, we need the gradient $\frac{\partial C}{\partial y_i}$.

Recall that $C = \sum_{k,l} p_{kl} \log p_{kl} - \sum_{k,l} p_{kl} \log q_{kl}$.
The first term is constant relative to $y$, so we only differentiate $-\sum p_{kl} \log q_{kl}$.

Let $d_{kl} = \|y_k - y_l\|$.
The low-D kernel is $w_{kl} = (1 + d_{kl}^2)^{-1}$.
Then $q_{kl} = \frac{w_{kl}}{Z}$, where $Z = \sum_{m \neq n} w_{mn}$.

The gradient derivation involves the chain rule:
$$
\frac{\partial C}{\partial y_i} = \sum_j \frac{\partial C}{\partial q_{ij}} \dots
$$

Skipping the tedious algebraic expansion, the result simplifies beautifully:

$$
\frac{\partial C}{\partial y_i} = 4 \sum_j (p_{ij} - q_{ij}) (y_i - y_j) (1 + \|y_i - y_j\|^2)^{-1}
$$

### Physical Interpretation of the Gradient
This equation represents a system of springs forces:
1.  **Attraction ($p_{ij}$ term):** If $p_{ij}$ (high-D similarity) is large but $q_{ij}$ (low-D similarity) is small, the term $(p_{ij} - q_{ij})$ is positive. This pulls $y_i$ toward $y_j$.
2.  **Repulsion ($q_{ij}$ term):** All points exert a small repulsive force on each other to prevent collapse.
3.  **T-Distribution Factor $(1 + \|y_i - y_j\|^2)^{-1}$:** This term comes from the derivative of the Student's t-distribution. It dampens the forces for points that are very far apart, allowing clusters to separate cleanly.

---

## 4. Algorithmic Steps for t-SNE

**Inputs:** Data $X$, Perplexity $Perp$, Iterations $T$, Learning Rate $\eta$, Momentum $\alpha(t)$.

1.  **Compute High-Dimensional Probabilities ($p_{ij}$):**
    * Calculate pairwise Euclidean distances $\|x_i - x_j\|^2$.
    * For each $i$, perform a binary search to find $\sigma_i$ such that the perplexity of the distribution matches $Perp$.
    * Compute $p_{j|i}$.
    * Compute symmetrized $p_{ij} = \frac{p_{j|i} + p_{i|j}}{2N}$.

2.  **Initialization:**
    * Initialize $y_i$ randomly (Gaussian with small variance $\sim 10^{-4}$) or using PCA (principal components).

3.  **Optimization Loop (for $t = 1$ to $T$):**
    * **A. Compute Low-D Affinities:**
        * Compute pairwise distances $\|y_i - y_j\|^2$.
        * Compute unnormalized weights $w_{ij} = (1 + \|y_i - y_j\|^2)^{-1}$.
        * Compute $Z = \sum_{k \neq l} w_{kl}$.
        * Compute $q_{ij} = w_{ij} / Z$.

    * **B. Compute Gradients:**
        $$
        \frac{\partial C}{\partial y_i} = 4 \sum_j (p_{ij} - q_{ij})(y_i - y_j)(1 + \|y_i - y_j\|^2)^{-1}
        $$

    * **C. Update Points (Gradient Descent with Momentum):**
        $$
        y_i^{(t)} = y_i^{(t-1)} - \eta \frac{\partial C}{\partial y_i} + \alpha(t) (y_i^{(t-1)} - y_i^{(t-2)})
        $$

    * **D. Heuristics (Optional but standard):**
        * **Early Exaggeration:** For the first ~250 iterations, multiply all $p_{ij}$ by a factor (e.g., 12). This forces tight clusters to form and move through each other easily initially.

**Output:** The final coordinates $Y$.

In [19]:
from manim import *
import numpy as np

# --- Configuration ---
config.frame_width = 16
config.frame_height = 9

class TSNEExplanation(Scene):
    def construct(self):
        # --- 0. Title and Overview ---
        self.title = Text("t-SNE: t-distributed Stochastic Neighbor Embedding", font_size=40).to_edge(UP)
        self.add(self.title) 
        self.wait(0.5)
        
        # Initial Subtitle
        self.current_subtitle = Text("Deep Dive: The Physics of High-Dimensional Mapping", font_size=32)
        self.current_subtitle.next_to(self.title, DOWN, buff=0.4).to_edge(LEFT, buff=1.0)
        self.play(Write(self.current_subtitle))
        self.wait(1)
        
        # --- 1. High-Dimensional Similarity (Binary Search for Sigma) ---
        self.section_1_perplexity_search()
        
        # --- 2. Low-Dimensional Similarity (Crowding) ---
        self.section_2_crowding()
        
        # --- 3. Objective Function (KL Divergence) ---
        self.section_3_kl_divergence()
        
        # --- 4. The Gradient (Force vs Distance Physics) ---
        self.section_4_gradient_physics()
        
        # --- 5. Optimization with Early Exaggeration ---
        self.section_5_optimization_phases()
        
        # Final cleanup
        self.play(FadeOut(self.title, self.current_subtitle))
        self.wait(1)

    # --- Helper: Smooth Subtitle Transition ---
    def transition_subtitle(self, new_text):
        new_subtitle = Text(new_text, font_size=32).next_to(self.title, DOWN, buff=0.4).to_edge(LEFT, buff=1.0)
        self.play(ReplacementTransform(self.current_subtitle, new_subtitle))
        self.current_subtitle = new_subtitle
        return new_subtitle

    # --- Section 1: Perplexity & Binary Search ---
    def section_1_perplexity_search(self):
        self.transition_subtitle("1. High-D: Adapting Sigma via Binary Search")

        # Setup: A dense cluster and a single point
        center = LEFT * 3
        
        # Safe Vector Math
        neighbors = VGroup(*[
            Dot(radius=0.08, color=BLUE).move_to(
                np.concatenate((np.random.normal(0, 0.5, 2), [0])) + center
            ) for _ in range(8)
        ])
        
        target_pt = Dot(radius=0.12, color=YELLOW).move_to(center)
        label_target = MathTex("x_i").next_to(target_pt, UP, buff=0.1)
        
        # Text explanation area (Right side)
        perp_text = Text("Perplexity = Desired # of Neighbors", font_size=24, color=YELLOW).to_edge(RIGHT, buff=2.0).shift(UP*1)
        search_text = Text("Binary Search for Sigma:", font_size=24).next_to(perp_text, DOWN, aligned_edge=LEFT)
        status_text = Text("Searching...", font_size=24, color=GRAY).next_to(search_text, DOWN, aligned_edge=LEFT)
        
        self.play(FadeIn(neighbors), FadeIn(target_pt), Write(label_target))
        self.play(Write(perp_text), Write(search_text))
        
        # Animation: Binary Search Visualization
        search_circle = Circle(radius=0.1, color=YELLOW, fill_opacity=0.1).move_to(center)
        self.add(search_circle)
        
        # Step 1: Too Small
        self.play(
            search_circle.animate.scale(5), # Radius 0.1 -> 0.5
            Transform(status_text, Text("Sigma too small (Entropy low)", font_size=24, color=RED).next_to(search_text, DOWN, aligned_edge=LEFT)),
            run_time=0.8
        )
        self.wait(0.2)
        
        # Step 2: Too Large
        self.play(
            search_circle.animate.scale(6), # Radius 0.5 -> 3.0
            Transform(status_text, Text("Sigma too large (Entropy high)", font_size=24, color=RED).next_to(search_text, DOWN, aligned_edge=LEFT)),
            run_time=0.8
        )
        self.wait(0.2)
        
        # Step 3: Just Right
        self.play(
            search_circle.animate.scale(0.5), # Radius 3.0 -> 1.5
            Transform(status_text, Text("Sigma converged! (Target met)", font_size=24, color=GREEN).next_to(search_text, DOWN, aligned_edge=LEFT)),
            run_time=0.8
        )
        
        # Formula display
        p_ji_eq = MathTex(
            r"p_{j|i} = \frac{\exp(-\|x_i - x_j\|^2 / 2\sigma_i^2)}{\sum \dots}"
        ).scale(0.8).to_edge(DOWN, buff=0.5)
        
        self.play(Write(p_ji_eq))
        self.wait(2)
        
        self.play(
            FadeOut(neighbors), FadeOut(target_pt), FadeOut(label_target), 
            FadeOut(search_circle), FadeOut(perp_text), FadeOut(search_text), 
            FadeOut(status_text), FadeOut(p_ji_eq)
        )

    # --- Section 2: Crowding Physics ---
    def section_2_crowding(self):
        self.transition_subtitle("2. Low-D: The 'Crowding' Volume Mismatch")

        # Visualization: Volume difference
        # 1. High Dimension Rep
        shell_3d_text = Text("High-D Space (Example: 100D)", font_size=24, color=BLUE).move_to(LEFT * 4 + UP * 2)
        shell_desc = Text("Volume grows exponentially.\nNeighbors are equidistant on a 'shell'.", font_size=20, color=GRAY).next_to(shell_3d_text, DOWN)
        
        # Create a ring of points
        ring_radius = 2.0
        ring_center = LEFT * 4 + DOWN * 0.5
        ring_pts = VGroup(*[
            Dot(radius=0.08, color=BLUE).move_to(
                np.array([np.cos(theta)*ring_radius, np.sin(theta)*ring_radius, 0]) + ring_center
            ) for theta in np.linspace(0, 2*np.pi, 12, endpoint=False)
        ])
        
        self.play(Write(shell_3d_text), Write(shell_desc), Create(ring_pts))

        # 2. Low Dimension Rep
        plate_2d_text = Text("Low-D Space (2D)", font_size=24, color=RED).move_to(RIGHT * 4 + UP * 2)
        plate_desc = Text("Area grows quadratically.\nNot enough room in the center!", font_size=20, color=GRAY).next_to(plate_2d_text, DOWN)
        
        # Create a "crushed" cluster
        crush_center = RIGHT * 4 + DOWN * 0.5
        crush_pts = VGroup(*[
            Dot(radius=0.08, color=RED).move_to(
                np.concatenate((np.random.normal(0, 0.5, 2), [0])) + crush_center
            ) for _ in range(12)
        ])
        
        self.play(Write(plate_2d_text), Write(plate_desc))
        
        # Transformation Arrow
        arrow = Arrow(start=LEFT*1, end=RIGHT*1, color=GRAY)
        self.play(Create(arrow), TransformFromCopy(ring_pts, crush_pts))
        
        # The Solution: Heavy Tail
        solution_text = Text("Solution: Student-t distribution pushes distant points away,", font_size=22, color=YELLOW).to_edge(DOWN, buff=1.0)
        solution_sub = Text("creating 'fake' space for neighbors.", font_size=22, color=YELLOW).next_to(solution_text, DOWN)
        
        self.play(Write(solution_text), Write(solution_sub))
        self.wait(2)
        
        self.play(FadeOut(shell_3d_text), FadeOut(shell_desc), FadeOut(ring_pts),
                  FadeOut(plate_2d_text), FadeOut(plate_desc), FadeOut(crush_pts),
                  FadeOut(arrow), FadeOut(solution_text), FadeOut(solution_sub))

    # --- Section 3: KL Divergence (Fixed Overlap) ---
    def section_3_kl_divergence(self):
        self.transition_subtitle("3. Objective: KL Divergence as Force Generator")

        # 1. Formula (High up)
        # Fix: Move UP to 2.0 to clear the columns
        kl_eq = MathTex(
            r"C = \sum p_{ij} \log \frac{p_{ij}}{q_{ij}}"
        ).scale(1.2).move_to(UP * 2.0)
        self.play(Write(kl_eq))

        # 2. Physics Interpretation Tables
        # Fix: Move columns DOWN to UP*0.5
        
        # Headers
        h_left = Text("Case A: Neighbors (High P)", font_size=26, color=BLUE).move_to(LEFT*4 + UP*0.5)
        h_right = Text("Case B: Distant (Low P)", font_size=26, color=RED).move_to(RIGHT*4 + UP*0.5)
        self.play(Write(h_left), Write(h_right))
        
        # Logic Chains
        # Left
        l1 = Text("Mapped far apart (Low Q)?", font_size=20).next_to(h_left, DOWN)
        l2 = MathTex(r"\log(P/Q) \gg 0").scale(0.8).next_to(l1, DOWN)
        l3 = Text("High Cost -> Attraction Force", font_size=20, color=YELLOW).next_to(l2, DOWN)
        
        # Right
        r1 = Text("Mapped close together (High Q)?", font_size=20).next_to(h_right, DOWN)
        r2 = MathTex(r"P \approx 0 \implies P \log(\dots) \approx 0").scale(0.8).next_to(r1, DOWN)
        r3 = Text("Low Cost -> Minimal Influence", font_size=20, color=GRAY).next_to(r2, DOWN)
        
        self.play(Write(l1), Write(l2), Write(l3))
        self.play(Write(r1), Write(r2), Write(r3))
        
        # 3. Force Visualization (Bottom Zone)
        # Fix: Use explicit DOWN coordinate to avoid text
        anim_y = DOWN * 2.5
        
        # Attraction Visualization
        p1 = Dot(color=BLUE).move_to(LEFT * 4 + anim_y)
        p2 = Dot(color=BLUE).move_to(LEFT * 2 + anim_y)
        
        # Spring connecting them
        spring = Line(p1.get_center(), p2.get_center(), color=YELLOW).set_stroke(width=4)
        lbl_spring = Text("Strong Pull", font_size=20).next_to(spring, UP)
        
        self.play(FadeIn(p1), FadeIn(p2), Create(spring), Write(lbl_spring))
        self.play(
            p1.animate.shift(RIGHT*0.8), 
            p2.animate.shift(LEFT*0.8), 
            spring.animate.scale(0.2),
            run_time=1
        )
        
        self.wait(1)
        
        # Cleanup
        self.play(
            FadeOut(kl_eq), FadeOut(h_left), FadeOut(h_right),
            FadeOut(l1), FadeOut(l2), FadeOut(l3),
            FadeOut(r1), FadeOut(r2), FadeOut(r3),
            FadeOut(p1), FadeOut(p2), FadeOut(spring), FadeOut(lbl_spring)
        )

    # --- Section 4: The Physics of the Gradient (Fixed Overlap) ---
    def section_4_gradient_physics(self):
        self.transition_subtitle("4. Minimization: Why t-SNE allows separation")
        
        # 1. The equation (Layout Fix: Scale down and move up)
        grad_eq = MathTex(
            r"\frac{\partial C}{\partial y_i} \propto \sum (p_{ij} - q_{ij}) (y_i - y_j) (1 + \|y_i - y_j\|^2)^{-1}"
        ).scale(0.7).move_to(UP * 2.0)
        self.play(Write(grad_eq))
        
        # 2. Labels (Layout Fix: Stagger positions)
        
        # Term 1: Net Attraction (P - Q) -> Place ABOVE
        # Index range approx for (p_ij - q_ij)
        rect_mag = SurroundingRectangle(grad_eq[0][12:21], color=RED, buff=0.1)
        lbl_mag = Text("Net Attraction", color=RED, font_size=18).next_to(rect_mag, UP)
        
        # Term 2: Direction (y - y) -> Place BELOW
        rect_dir = SurroundingRectangle(grad_eq[0][-21:-14], color=BLUE, buff=0.1)
        lbl_dir = Text("Direction", color=BLUE, font_size=18).next_to(rect_dir, DOWN)
        
        # Term 3: Stiffness (1 + dist)^-1 -> Place BELOW but verify x-clearance
        # This is the last term
        stiff_part = grad_eq[0][-13:]
        rect_stiff = SurroundingRectangle(stiff_part, color=GREEN, buff=0.1)
        lbl_stiff = Text("Inverse Distance Law", color=GREEN, font_size=18).next_to(rect_stiff, DOWN)
        
        self.play(Create(rect_mag), Write(lbl_mag))
        self.play(Create(rect_dir), Write(lbl_dir))
        self.play(Create(rect_stiff), Write(lbl_stiff))
        
        # 3. Graph (Layout Fix: Ensure text doesn't overlap equation)
        axes = Axes(
            x_range=[0, 5, 1],
            y_range=[0, 0.6, 0.2],
            x_length=6,
            y_length=3,
            axis_config={"include_numbers": False}
        ).move_to(DOWN * 1.5)
        
        x_label = axes.get_x_axis_label("Distance ||y_i - y_j||")
        y_label = axes.get_y_axis_label("Attraction Force")
        
        graph_tsne = axes.plot(lambda x: x / (1 + x**2), color=YELLOW, x_range=[0, 5])
        
        # Fix: Move label to right to avoid center clutter
        lbl_tsne = Text("t-SNE Force", font_size=18, color=YELLOW).next_to(axes.c2p(1, 0.5), UP).shift(RIGHT*0.5)
        
        self.play(Create(axes), Write(x_label), Write(y_label))
        self.play(Create(graph_tsne), Write(lbl_tsne))
        
        # Decay Arrow
        decay_arrow = Arrow(start=axes.c2p(2, 0.4), end=axes.c2p(4, 0.2), color=YELLOW)
        decay_text = Text("Force DIES OUT for outliers!", font_size=24, color=YELLOW).next_to(decay_arrow, UP)
        
        self.play(Create(decay_arrow), Write(decay_text))
        
        self.wait(3)
        
        self.play(
            FadeOut(grad_eq), FadeOut(rect_mag), FadeOut(lbl_mag),
            FadeOut(rect_dir), FadeOut(lbl_dir), FadeOut(rect_stiff), FadeOut(lbl_stiff),
            FadeOut(axes), FadeOut(x_label), FadeOut(y_label),
            FadeOut(graph_tsne), FadeOut(lbl_tsne),
            FadeOut(decay_arrow), FadeOut(decay_text)
        )

    # --- Section 5: Optimization with Early Exaggeration (Fixed Overlap) ---
    def section_5_optimization_phases(self):
        self.transition_subtitle("5. Optimization: Early Exaggeration & Momentum")
        
        # Setup points
        np.random.seed(99)
        clusters = [
            (np.array([-3, -1, 0]), BLUE),
            (np.array([3, -1, 0]), RED)
        ]
        
        points = VGroup()
        destinations = []
        
        # Create scattered start
        for center, col in clusters:
            for _ in range(10):
                start_pos = np.concatenate((np.random.uniform(-4, 4, 2), [0])) + np.array([0, -1, 0])
                p = Dot(radius=0.08, color=col).move_to(start_pos)
                points.add(p)
                destinations.append(np.concatenate((np.random.normal(0, 0.5, 2), [0])) + center)
        
        self.play(FadeIn(points))
        
        # PHASE 1: Early Exaggeration
        # Fix: Position relative to subtitle to avoid header collision
        phase1_text = Text("Phase 1: Early Exaggeration (P multiplied by 4)", font_size=24, color=ORANGE)\
            .next_to(self.current_subtitle, DOWN, buff=0.5)
        
        force_indicator = Text("STRONG FORCES ACTIVATED", font_size=36, color=RED).move_to(ORIGIN).set_opacity(0.5)
        
        self.play(Write(phase1_text))
        self.play(FadeIn(force_indicator), run_time=0.5)
        self.play(FadeOut(force_indicator), run_time=0.5)
        
        # Fast initial movement (Explosion/Organization)
        anims_p1 = []
        midpoints = []
        for dot, dest in zip(points, destinations):
            mid = (dot.get_center() + dest) / 2 + np.concatenate((np.random.normal(0, 0.5, 2), [0])) # Chaos
            midpoints.append(mid)
            anims_p1.append(dot.animate.move_to(mid))
            
        self.play(*anims_p1, run_time=1.5, rate_func=rush_into)
        
        # PHASE 2: Fine Tuning
        phase2_text = Text("Phase 2: Fine Tuning (Momentum decay)", font_size=24, color=GREEN)\
            .next_to(self.current_subtitle, DOWN, buff=0.5)
        
        self.play(Transform(phase1_text, phase2_text))
        
        # Slower final convergence
        anims_p2 = []
        for dot, dest in zip(points, destinations):
            anims_p2.append(dot.animate.move_to(dest))
            
        self.play(*anims_p2, run_time=2.5, rate_func=exponential_decay)
        
        # Final Label
        final_lbl = Text("Global clusters separated, Local structure kept.", font_size=24, color=YELLOW).to_edge(DOWN, buff=1.0)
        self.play(Write(final_lbl))
        self.wait(2)
        
        self.play(FadeOut(points), FadeOut(phase1_text), FadeOut(final_lbl))


%manim -qk -v warning TSNEExplanation

Manim Community v0.19.0